# Titanic - missing values
**Mateusz Michałek 426994**

**Cel ćwiczenia:** Identyfikacja braków danych, klasyfikacja mechanizmu ich powstawania (MCAR, MAR, MNAR) oraz dobór odpowiedniej strategii postępowania z brakującymi wartościami.

In [12]:

import numpy as np
import pandas as pd


def load_arff_to_df(filepath: str) -> pd.DataFrame:
    """Wczytuje plik .arff do obiektu pandas.DataFrame."""
    attributes = []
    data_start = 0

    with open(filepath, "r", encoding="utf-8") as file:
        for idx, line in enumerate(file):
            stripped = line.strip()
            lower_line = stripped.lower()
            if lower_line.startswith("@attribute"):
                col_name = stripped.split()[1].strip("'\"")
                attributes.append(col_name)
            elif lower_line.startswith("@data"):
                data_start = idx + 1
                break

    return pd.read_csv(
        filepath,
        skiprows=data_start,
        header=None,
        names=attributes,
        quotechar='"',
        skipinitialspace=True,
    )


FILE_PATH = "Titanic.arff"
df_raw = load_arff_to_df(FILE_PATH)

## 1. Wstępna inspekcja danych (pierwsze 20 wierszy)

In [13]:
print(f"Liczba wierszy (obserwacji): {df_raw.shape[0]}")
print(f"Liczba kolumn (cech łącznie ze zmienną docelową): {df_raw.shape[1]}")
print(f"Lista kolumn: {list(df_raw.columns)}")

df_raw.head(20)

Liczba wierszy (obserwacji): 1309
Liczba kolumn (cech łącznie ze zmienną docelową): 14
Lista kolumn: ['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest']


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29,0,0,24160,211.3375,B5,S,2,?,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.55,C22 C26,S,11,?,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2,1,2,113781,151.55,C22 C26,S,?,?,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30,1,2,113781,151.55,C22 C26,S,?,135,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25,1,2,113781,151.55,C22 C26,S,?,?,"Montreal, PQ / Chesterville, ON"
5,1,1,"Anderson, Mr. Harry",male,48,0,0,19952,26.55,E12,S,3,?,"New York, NY"
6,1,1,"Andrews, Miss. Kornelia Theodosia",female,63,1,0,13502,77.9583,D7,S,10,?,"Hudson, NY"
7,1,0,"Andrews, Mr. Thomas Jr",male,39,0,0,112050,0,A36,S,?,?,"Belfast, NI"
8,1,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",female,53,2,0,11769,51.4792,C101,S,D,?,"Bayside, Queens, NY"
9,1,0,"Artagaveytia, Mr. Ramon",male,71,0,0,PC 17609,49.5042,?,C,?,22,"Montevideo, Uruguay"


Zbiór zawiera 14 kolumn:

*   `survived` - kolumna binarna, opisująca czy dany pasażer przeżył - wartość 1, czy nie przeżył - wartość 0,
*   kolumny `name`, `sex`, `age` opisują podstawowe dane osobowe pasażerów,
*   `sibsp` oraz `parch` opisują zalezności rodzinne wśród pasażerów,
*   `pclass`, `ticker`, `fare`, `cabin`, `embarked` zawierają informacje o podróży, takie jak klasa, numer biletu, czy kabina,
*   `boat` i  `body` opisują informacje powypadkowe,
*   `home.dest` opisuje miejsce zamieszkania i cel podróży pasażera.

Na pierwszy rzut oka widać zaelżność między tym, czy dany pasażer miał miejsce w łodzi ratunkowej, a tym czy przeżył (survived == 1).




## 2. Analiza ilościowa brakujących wartości

**Czy zbiór danych należy zmodyfikować przed użyciem funkcji `isnull()`?**

**Tak.** W formacie .arff braki danych oznaczane są znakiem `?`. Funkcje `pd.isnull()` rozpoznają wyłącznie wartości `NaN` / `None`, dlatego przed ich wywołaniem należy zamienić wszystkie znaki `?` na `np.nan` oraz przywrócić kolumnom liczbowym właściwy typ numeryczny.

In [14]:
# Sprawdzenie przed modyfikacją
print("Braki przed zamianą '?' na NaN:", df_raw.isnull().sum().sum())

#zamiana '?' na np.nan i zamiana na typy numeryczne
df = df_raw.replace("?", np.nan)
numeric_cols = ["pclass", "survived", "age", "sibsp", "parch", "fare", "body"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

missing_summary = pd.DataFrame(
    {
        "liczba_brakow": df.isnull().sum(),
        "udzial_brakow": df.isnull().mean(),
        "procent_brakow [%]": (df.isnull().mean() * 100).round(2),
    }
).sort_values(by="udzial_brakow", ascending=False)

missing_summary

Braki przed zamianą '?' na NaN: 0


,liczba_brakow,udzial_brakow,procent_brakow [%]
body,1188,0.907563,90.76
cabin,1014,0.774637,77.46
boat,823,0.628724,62.87
home.dest,564,0.430863,43.09
age,263,0.200917,20.09
embarked,2,0.001528,0.15
fare,1,0.000764,0.08
sibsp,0,0.000000,0.00
name,0,0.000000,0.00
survived,0,0.000000,0.00


* Po konwersji `?` na `np.nan` widać istotne zróżnicowanie udziału braków pomiędzy zmiennymi:
  * **`body` (~91%)** – najwyższy odsetek braków; numer identyfikacyjny odnalezionego ciała dotyczy wyłącznie części ofiar katastrofy.
  * **`cabin` (~77%)** – brak numeru kabiny u większości pasażerów (szczególnie w 2. i 3. klasie).
  * **`boat` (~63%)** – brak numeru szalupy ratunkowej dla osób, które nie zostały ewakuowane.
  * **`home.dest` (~43%)** – niepełne dane o miejscu pochodzenia/docelowym.
  * **`age` (~20%)** – istotna część pasażerów nie ma odnotowanego wieku.
  * **`embarked` (<1%) oraz `fare` (<1%)** – pojedyncze, śladowe braki.
* Pozostałe zmienne (`pclass`, `survived`, `name`, `sex`, `sibsp`, `parch`, `ticket`) są kompletne (0% braków).

## 3. Identyfikacja typu brakujących danych (MCAR, MAR, MNAR)

In [15]:
# Wariant dwuliniowy (wg wskazówki) na przykładzie kolumny 'cabin'
df["cabin_null"] = np.where(df["cabin"].isnull(), 1, 0)
print("Brak 'cabin' w zależności od 'survived' (2 linijki):")
print(df.groupby(["survived"])["cabin_null"].mean())

# Wersja jednoliniowa
print("\nBrak 'cabin' w zależności od 'survived' (1 linijka):")
print(df["cabin"].isnull().groupby(df["survived"]).mean())

# Zbiorcze zestawienie odsetka braków względem 'survived' dla wszystkich cech
cols_with_missing = [
    col for col in df.columns
    if col != "cabin_null" and df[col].isnull().sum() > 0
]

missing_vs_survived = (
    df[cols_with_missing]
    .isnull()
    .groupby(df["survived"])
    .mean()
    .T
    .round(4)
)
missing_vs_survived.columns = ["Brak dla survived=0", "Brak dla survived=1"]
missing_vs_survived

Brak 'cabin' w zależności od 'survived' (2 linijki):
survived
0    0.873918
1    0.614000
Name: cabin_null, dtype: float64

Brak 'cabin' w zależności od 'survived' (1 linijka):
survived
0    0.873918
1    0.614000
Name: cabin, dtype: float64


,Brak dla survived=0,Brak dla survived=1
age,0.2349,0.146
fare,0.0012,0.000
cabin,0.8739,0.614
embarked,0.0000,0.004
boat,0.9889,0.046
body,0.8504,1.000
home.dest,0.5080,0.306


Zestawienie braków względem zmiennej `survived` pokazuje silne powiązanie występowania wartości `NaN` z faktem przeżycia katastrofy:
* **`boat` i `body` – MNAR (Missing Not at Random):**
  Brak danych wynika bezpośrednio z mechanizmu przyczynowo-skutkowego samej katastrofy. Osoby, które nie przeżyły (`survived=0`), niemal w 100% nie mają przypisanego numeru szalupy (`boat` jest `NaN`), natomiast wśród ocalałych (`survived=1`) numer ciała (`body`) jest pusty w 100% przypadków.
* **`cabin`, `age`, `home.dest` – MAR (Missing at Random):**
  Występowanie braków jest wyraźnie wyższe wśród ofiar (`survived=0`) niż wśród ocalałych (`survived=1`) oraz wiąże się z innymi obserwowanymi zmiennymi (np. z klasą podróży `pclass`). Listy pasażerów 3. klasy były mniej kompletne, a od osób, które przeżyły, łatwiej było uzupełnić dane (wiek, miejsce docelowe) po katastrofie.
* **`embarked`, `fare` – MCAR (Missing Completely at Random):**
  Pojedyncze braki mają charakter losowych błędów lub przeoczeń w dokumentacji rejsu i nie wykazują istotnego związku z przeżyciem.

## 4. Rekomendowany sposób postępowania z brakującymi wartościami

Ze względu na różny mechanizm powstawania braków oraz ich udział procentowy należy zastosować strategię dopasowaną do konkretnej cechy:

1. **`boat` oraz `body` – usunięcie kolumn przed modelowaniem:**
   Są to zmienne powypadkowe (skutek katastrofy, a nie cecha pasażera przed wypłynięciem). Pozostawienie ich w modelu predykcyjnym przeżycia powodowałoby wyciek danych (*data leakage*).
2. **`cabin` (~77% braków) – inżynieria cech (kodowanie braku jako osobnej kategorii):**
   Usunięcie wierszy spowodowałoby utratę większości zbioru. Ponieważ sam brak numeru kabiny jest skorelowany z `pclass` i `survived`, warto wyodrębnić pierwszą literę (pokład), a brak zastąpić kategorią `'Unknown'` (lub utworzyć cechę binarną `has_cabin`).
3. **`age` (~20% braków) – imputacja warunkowa + wskaźnik braku:**
   Zamiast globalnej średniej lepiej uzupełnić braki medianą wyznaczoną w podgrupach (np. wg `pclass` i `sex` lub tytułu z `name`) albo metodą `KNNImputer` / `IterativeImputer`. Dodatkowo można zachować binarną kolumnę informującą o pierwotnym braku wieku (`age_isnull`).
4. **`home.dest` (~43% braków) – zastąpienie kategorią `'Unknown'` lub usunięcie:**
   Ze względu na duży odsetek braków i bardzo wysoką liczbę unikalnych tekstów, do klasycznych modeli kolumnę warto usunąć albo zagregować (np. do kraju) z wartością `'Unknown'` dla braków.
5. **`embarked` i `fare` (<1% braków) – prosta imputacja:**
   Pojedyncze braki można bezpiecznie uzupełnić dominantą (`embarked`) oraz medianą dla danej klasy biletu (`fare`).
